# ⚙️ Configuration & Export

How to configure parsing, mask proper nouns, and build multi-project datasets.

## 🎯 Goals of this notebook

1. **Understand RunConfig options** — learn every toggle the parser exposes
2. **Compare outputs** — see how dropping damaged signs or masking POS changes transliterations
3. **Build a multi-project dataset** — parse several corpora and combine into one DataFrame
4. **Export** — save combined data as JSONL and CSV for downstream use
5. **Quick analysis** — explore the combined dataset by project, provenance, and period

## Configuration Options

Before we start, if the default data paths aren't correct for your machine, you can re-configure them:
By default, `DATA_DIR` is set to `<repo_root>/enriched_data`. If you have it somewhere else (like `D:/thesis_data`):

In [1]:
import oracc_parser.settings as settings
from pathlib import Path

# settings.DATA_DIR = Path("D:/my_custom_oracc_data")

## 1. RunConfig options

| Option | Default | What it does |
|---|---|---|
| `limit` | `None` | Only parse first N texts (good for testing) |
| `drop_missing` | `False` | Remove signs marked `[x]` (completely broken) |
| `drop_damaged` | `False` | Remove signs marked `⸢x⸣` (partially damaged) |
| `mask_pos` | `[]` | Replace words of certain POS with `[TAG]` |
| `languages` | `["Akkadian"]` | Which languages to process |
| `USE_CACHE` | `True` | Use cached results if available |

### Discovering Valid Configurations
If you want to know what values are valid for `mask_pos`, `languages`, or `periods`, you can query the bundled reference data directly:

In [2]:
from oracc_parser.pipeline import reference_data

# See valid POS tags for masking
pos_df = reference_data.get_pos_tags()
print('Valid POS tags (first 15):')
print(pos_df['POS-tag'].head(15).tolist())
print()

# See valid languages
lang_df = reference_data.get_languages()
print('Valid Languages:')
print(lang_df['lang'].tolist())


Valid POS tags (first 15):
['MN', 'n', 'u', 'N', 'CN', 'AJ', 'PRP', 'V', 'IP', 'PN', 'MOD', 'RN', 'GN', 'REL', 'AV']

Valid Languages:
['sux', 'akk', 'akk-x-neoass', 'akk-x-stdbab', 'akk-x-neobab', 'akk-x-midass', 'akk-x-mbperi', 'qpc', 'akk-x-ltebab', 'akk-x-oldbab', 'akk-949', 'uga-040', 'sux-x-emesal', 'xur', 'peo', 'xur-946', 'hit', 'akk-x-midbab', 'arc', 'arc-949', 'elx', 'akk-x-stdbab-949', 'akk-x-earakk', 'akk-x-oldass', 'uga', 'xhu', 'hit-946', 'akk-936', 'xur-944', 'qca', 'qeb', 'akk-x-mbperi-949', 'akk-x-oldakk', 'qcu', 'qpe', 'grc', 'qur', 'akk-935', 'xhu-040', 'akk-x-neoass-949', 'xhu-946', 'sux-947', 'egy-020', 'sux-x-gloss', 'hlu', 'akk-x-neobab-949', 'sux-x-syll', 'hit-040', 'akk-x-oldbab-949']


In [3]:
from oracc_parser import parse_project, RunConfig, get_transliterations

# Compare default vs. cleaned
PROJECT = "cams/gkab"

# Default: keep everything
rec_default = parse_project(PROJECT, config=RunConfig(limit=1))
print("DEFAULT:")
for _, r in get_transliterations(rec_default).iterrows():
    print(f"  {r['id']}: {r['transliteration'][:100]}")

print()

# Cleaned: drop broken signs, mask personal + divine names
rec_clean = parse_project(PROJECT, config=RunConfig(
    limit=None,
    drop_missing=True,
    mask_pos=[ "GN",  # Geographical Name
            "SN",  # State/City Name
            "TN",  # Temple Name
            "WN",  # Water Name
            "QN",  # Quarter Name
            "ON",  # Geographical Name (variant)
            "DN",  # Divine Name (Patron gods give away the city)
            "RN",  # Royal Name (Kings give away the city/era)ת
            "EN"],
))
print("CLEANED (drop broken, mask PN+DN):")
for _, r in get_transliterations(rec_clean).iterrows():
    print(f"  {r['id']}: {r['transliteration'][:100]}")

00:59:13 | INFO    | oracc_parser | Already downloaded: C:\Users\shaha\PycharmProjects\thesis\oracc_preprocessing\oracc-parser\enriched_data\jsonzip\cams-gkab.zip
Parsing cams/gkab: 100%|██████████| 1/1 [00:00<00:00, 102.26it/s]
00:59:16 | INFO    | oracc_parser | Loaded 1/1 tablets from cache, parsed 0 new
00:59:16 | INFO    | oracc_parser | Already downloaded: C:\Users\shaha\PycharmProjects\thesis\oracc_preprocessing\oracc-parser\enriched_data\jsonzip\cams-gkab.zip


DEFAULT:
  cams/gkab_P348427: ⸢NAM⸣.BUR₂ HUL ⸢MUŠ ša₂⸣ ina {iti}⸢BARA₂⸣ [...]
NAM.BUR₂.BI HUL MUŠ ša₂ ina E₂ ⸢LU₂⸣ [...]
⸢ana⸣ HUL



Parsing cams/gkab: 100%|██████████| 585/585 [00:17<00:00, 33.25it/s]
00:59:37 | INFO    | oracc_parser | Loaded 585/585 tablets from cache, parsed 0 new


CLEANED (drop broken, mask PN+DN):
  cams/gkab_P348427: ⸢NAM⸣.BUR₂ HUL ⸢MUŠ ša₂⸣ ina {iti}⸢BARA₂⸣ [...]
NAM.BUR₂.BI HUL MUŠ ša₂ ina E₂ ⸢LU₂⸣ [...]
⸢ana⸣ HUL
  cams/gkab_P348485: [...] ⸢EN⸣.TE.NA 3 NINDA KAŠ ŠE SUMUN ina IZI x [...]
[...] NAG-šu₂ ina li-la-a-tu₂ EGIR ša₂ NINDA-H
  cams/gkab_P363686: [... an]-⸢na⸣-a-ti ⸢ana⸣ DUH.LAL₃
[...] {na₄}GIŠ.NU₁₁.GAL
[...] HA SU x [...] x KI KAL E₂.GAL MAH DN
  cams/gkab_P363488: [...] 15 [...]
[...] 15 2.30 GAR ⸢ni⸣-ziq-⸢tu₄⸣ [...]
[...] x 15-⸢šu₂⸣ GAR x x TUR x x [...]
[...] x
  cams/gkab_P348835: 
  cams/gkab_P348761: MU 44-KAM RN LUGAL SIG 20 GUB ina MU.AN.NA 11 u₄-mu x x
1 11 11 u 16 27 ina MU 45-KAM SIG 27 20 GUB

  cams/gkab_P348675: [...] ⸢a-a⸣ {d}[...]
[...] ⸢HI⸣ {d}en ama ⸢a-a⸣ DN x [...] x x RI x [...]
[...] ⸢be⸣-lu-u₂ AD u AMA 
  cams/gkab_P348620: [...] x ⸢HUL-MEŠ ša⸣ [...]
[... it-ta]-⸢nab⸣-šu-u₂ it-ta-nam-[ma-ru]
[... lu]-u₂ pa-as-sa-[nik-ka]
⸢
  cams/gkab_P338350: 
  cams/gkab_P338686: I₃-MEŠ ⸢ina gul-gu⸣-le-⸢e lu tab⸣-[ku]
⸢SIG₂⸣-

## 2. POS masking reference

| Tag | Meaning 
|---|---|
| `PN` | Personal Name
| `DN` | Divine Name
| `GN` | Geographical Name 
| `RN` | Royal Name 
| `TN` | Temple Name

## 3. Build a multi-project dataset

In [4]:
import pandas as pd
from pathlib import Path
from oracc_parser import get_full_flat_table, get_metadata_table
from oracc_parser.settings import DATA_DIR, OUTPUT_DIR

# Find all downloaded projects
zip_dir = DATA_DIR / "jsonzip"
if not zip_dir.exists():
    zip_dir = Path("../data/jsonzip")

downloaded = sorted([f.stem for f in zip_dir.glob("*.zip")]) if zip_dir.exists() else []
print(f"Found {len(downloaded)} downloaded projects")
print(f"First 10: {downloaded[:10]}")

Found 149 downloaded projects
First 10: ['adsd', 'adsd-adart1', 'adsd-adart2', 'adsd-adart3', 'adsd-adart5', 'adsd-adart6', 'aemw', 'aemw-alalakh-idrimi', 'aemw-amarna', 'aemw-ugarit']


In [6]:
# Parse a few projects and combine
PROJECTS = ["cams/gkab"]  # change these to what you want
config = RunConfig(limit=3)  # small limit for demo — remove for full parse

all_dfs = []
for project in PROJECTS:
    print(f"Parsing {project}...")
    try:
        records = parse_project(project, config=config)
        df = get_full_flat_table(records)
        all_dfs.append(df)
        print(f"  → {len(records)} tablets")
    except Exception as e:
        print(f"  → Error: {e}")

combined = pd.concat(all_dfs, ignore_index=True)
print(f"✓ Combined: {len(combined)} tablets from {len(PROJECTS)} projects")
display(combined)

00:59:50 | INFO    | oracc_parser | Already downloaded: C:\Users\shaha\PycharmProjects\thesis\oracc_preprocessing\oracc-parser\enriched_data\jsonzip\cams-gkab.zip


Parsing cams/gkab...


Parsing cams/gkab: 100%|██████████| 3/3 [00:00<00:00, 60.87it/s]
00:59:53 | INFO    | oracc_parser | Loaded 3/3 tablets from cache, parsed 0 new


  → 3 tablets
✓ Combined: 3 tablets from 1 projects


,id,project,text_id,genre,archive,provenance,period,start_year,end_year,transliteration,normalization,lemmatization,unicode,translation,total_tokens,tokens_without_broken
0,cams/gkab_P348427,cams/gkab,P348427,Incantation-Ritual,Uncertain,Uruk,Uncertain,NaN,NaN,⸢NAM⸣.BUR₂ HUL ⸢MUŠ ša₂⸣ ina {iti}⸢BARA₂⸣ [......,namburbi lumun ṣerri ša ina Nisannu UNK\nnambu...,namburbû lumnu ṣerru ša ina Nisannu UNK\nnambu...,𒉆𒁔 𒅆𒌨 𒈲 𒃻 𒀸 𒌗𒁈 x\n𒉆𒁔𒁉 𒅆𒌨 𒈲 𒃻 𒀸 𒂍 𒇽 x\n𒁹 𒅆𒌨 𒌔 𒊒...,,283,270
1,cams/gkab_P348485,cams/gkab,P348485,Medical,Uncertain,Uruk,achaemenid,-550.0,-330.0,[...] ⸢EN⸣.TE.NA 3 NINDA KAŠ ŠE SUMUN ina IZI ...,UNK kaṣû UNK akal šikari uṭṭati labīri ina išā...,UNK kaṣû UNK akalu šikaru uṭṭatu labīru ina iš...,x 𒂗𒋼𒈾 𒐈 𒃻 𒁉 𒊺 𒌀 𒀸 𒉈 x x\nx 𒅘𒋙 𒀸 𒇷𒆷𒀀𒌓 𒂕 𒃻 𒃻𒄭𒀀 x...,"[...] cold, 3 akalu of old barley-beer on the ...",166,153
2,cams/gkab_P363686,cams/gkab,P363686,Astrological,,Uruk,Seleucid,-312.0,-64.0,[... an]-⸢na⸣-a-ti ⸢ana⸣ DUH.LAL₃\n[...] {na₄}...,UNK annâti ana iškūri\nUNK gišnugalli\nUNK UNK...,UNK annû ana iškūru\nUNK gišnugallu\nUNK UNK U...,x 𒀭𒈾𒀀𒋾 𒁹 𒂃𒋭\nx 𒎎𒄑𒉢𒃲\nx 𒄩 𒋢 x x x 𒆠 𒆗 𒂍𒃲 𒈤 𒀭𒈗𒄊𒊏...,[...] these [...] for wax [...] alabaster.\n[....,760,738


In [8]:
# Export the combined dataset
out = OUTPUT_DIR
out.mkdir(parents=True, exist_ok=True)

path_jsonl = out / "combined_dataset.jsonl"
path_csv = out / "combined_dataset.csv"

combined.to_json(path_jsonl, orient="records", lines=True, force_ascii=False)
combined.to_csv(path_csv, index=False)

print(f"Saved to:")
print(f"  {path_jsonl}  ({path_jsonl.stat().st_size/1024:.1f} KB)")
print(f"  {path_csv}  ({path_csv.stat().st_size/1024:.1f} KB)")
print(f"📁 All output files in {out}:")
for f in sorted(out.iterdir()):
    if f.is_file():
        print(f"  {f.name:40s}  {f.stat().st_size/1024:.1f} KB")

Saved to:
  C:\Users\shaha\PycharmProjects\thesis\oracc_preprocessing\oracc-parser\enriched_data\output\combined_dataset.jsonl  (41.1 KB)
  C:\Users\shaha\PycharmProjects\thesis\oracc_preprocessing\oracc-parser\enriched_data\output\combined_dataset.csv  (39.9 KB)
📁 All output files in C:\Users\shaha\PycharmProjects\thesis\oracc_preprocessing\oracc-parser\enriched_data\output:
  combined_dataset.csv                      39.9 KB
  combined_dataset.jsonl                    41.1 KB
  saao_saa01.csv                            26.8 KB
  saao_saa01.jsonl                          28.2 KB


## 4. Quick analysis of the data

In [9]:
print("Texts by project:")
print(combined["project"].value_counts().to_string())

print("Texts by provenance:")
print(combined["provenance"].value_counts().to_string())

print("Texts by period:")
print(combined["period"].value_counts().to_string())

print(f"Avg tokens per text: {combined['total_tokens'].mean():.0f}")
print(f"Total tokens: {combined['total_tokens'].sum()}")

Texts by project:
project
cams/gkab    3
Texts by provenance:
provenance
Uruk    3
Texts by period:
period
Uncertain     1
achaemenid    1
Seleucid      1
Avg tokens per text: 403
Total tokens: 1209
